<a href="https://colab.research.google.com/github/majaVtech/business-sales-analysis/blob/main/Notebooks/01_sales_analysisipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data inspection

The first review on datase.

In [1]:
import pandas as pd
url = ("https://raw.githubusercontent.com/majaVtech/business-sales-analysis/main/Data/raw_sales_data.csv")
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()
df.info()
df.isna().sum()
print("Duplicate rows:", df.duplicated().sum())
df.describe()
for column in ["product", "category", "city", "payment_method"]:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))

Shape: (20050, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20050 entries, 0 to 20049
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   order_id        20050 non-null  object 
 1   order_date      20050 non-null  object 
 2   customer_id     20050 non-null  object 
 3   customer_name   20050 non-null  object 
 4   product         20050 non-null  object 
 5   category        20050 non-null  object 
 6   quantity        20050 non-null  int64  
 7   unit_price      20050 non-null  float64
 8   discount        20050 non-null  float64
 9   city            19950 non-null  object 
 10  payment_method  19974 non-null  object 
dtypes: float64(2), int64(1), object(8)
memory usage: 1.7+ MB
Duplicate rows: 50

--- product ---
product
Wireless Mouse         2976
Coffee Maker           2012
Mechanical Keyboard    1962
Air Fryer              1626
Office Chair           1591
Bluetooth Speaker      1576
Desk Lamp    

## Data Quality Assessment

The initial assessment of the Northstar Retail sales dataset identified several data quality issues that should be addressed before performing the business analysis.

The dataset contains 20,050 records and 11 columns. Although most fields are complete, several inconsistencies were identified:

* 50 complete duplicate records were found.
* 100 records have a missing city value.
* 76 records have a missing payment method.
* Category names contain inconsistent capitalization, resulting in duplicate representations of the same categories.
* The `order_date` column is currently stored as a text field rather than a datetime field.
* Additional validation is required for numerical fields such as `quantity` and `unit_price` to identify potentially invalid negative values.

These issues could affect the accuracy of revenue calculations, category analysis, customer segmentation, and other business KPIs. The next step is therefore to clean and standardize the dataset before conducting the analysis.


## Data inspection

Checking for missing values.

In [2]:
df.isna().sum()

,0
order_id,0
order_date,0
customer_id,0
customer_name,0
product,0
category,0
quantity,0
unit_price,0
discount,0
city,100


Chacking for duplicate rows.

In [3]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 50


Description of numeric columns in dataset.

In [4]:
df.describe()

,quantity,unit_price,discount
count,20050.000000,20050.00000,20050.000000
mean,1.753367,91.33793,0.052581
std,1.030518,97.13023,0.059914
min,-1.000000,-50.00000,0.000000
25%,1.000000,30.32250,0.000000
50%,1.000000,73.15500,0.050000
75%,2.000000,93.95000,0.100000
max,5.000000,524.98000,0.200000


Checking valuesa through columns of importance.

In [5]:
for column in ["product", "category", "city", "payment_method"]:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))


--- product ---
product
Wireless Mouse         2976
Coffee Maker           2012
Mechanical Keyboard    1962
Air Fryer              1626
Office Chair           1591
Bluetooth Speaker      1576
Desk Lamp              1477
Notebook               1411
Running Shoes          1197
Yoga Mat               1009
Skincare Set            966
Smartphone              809
Printer                 808
Hair Dryer              630
Name: count, dtype: int64

--- category ---
category
Electronics        7291
Home & Kitchen     3621
Furniture          3048
Office Supplies    2204
Sports             2194
Beauty             1592
ELECTRONICS          32
FURNITURE            20
HOME & KITCHEN       17
OFFICE SUPPLIES      15
SPORTS               12
BEAUTY                4
Name: count, dtype: int64

--- city ---
city
Banja Luka    4484
Sarajevo      3837
Mostar        2450
Tuzla         2015
Zenica        1563
Bijeljina     1396
Prijedor      1224
Brcko         1170
Doboj         1057
Trebinje       754
NaN    

## Data Cleanining

Starting with converting `order_date` from _object_ data type to _datetime_

In [6]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
print(df['order_date'].dtype)
print(df['order_date'].min())
print(df['order_date'].max())

datetime64[ns]
2025-01-01 00:00:00
2025-12-31 00:00:00


Standardizing `category` column to all lower cases.
From befor we have seen that we had same names of categories, but writen in difrent font size.

In [7]:
df['category'] = df['category'].str.strip().str.lower()
print(df['category'].value_counts(dropna=False))

category
electronics        7323
home & kitchen     3638
furniture          3068
office supplies    2219
sports             2206
beauty             1596
Name: count, dtype: int64


Remving duplicated values in dataset.

In [8]:
df.drop_duplicates(inplace=True)
print("Shape after dropping duplicates:", df.shape)
print("Duplicates remaining:", df.duplicated().sum())

Shape after dropping duplicates: (20000, 11)
Duplicates remaining: 0


Inspecting numerical columns.
Searching for negative values that can affect further analysis in negativ way.

In [12]:
numerical_columns = [
    "quantity",
    "unit_price",
    "discount"
]
print("Negative quantities:", (df['quantity'] < 0).sum())
print("Negative unit prices:", (df['unit_price'] < 0).sum())
print("Negative discounts:", (df['discount'] < 0).sum())
print("Minimum values:")
print(df[numerical_columns].min())

df[df['quantity'] < 0]

Negative quantities: 10
Negative unit prices: 5
Negative discounts: 0
Minimum values:
quantity      -1.0
unit_price   -50.0
discount       0.0
dtype: float64


,order_id,order_date,customer_id,customer_name,product,category,quantity,unit_price,discount,city,payment_method
1240,ORD02912,2025-01-22,CUST0022,Customer 0022,Skincare Set,beauty,-1,40.46,0.10,Banja Luka,Bank Transfer
6302,ORD17212,2025-04-25,CUST0139,Customer 0139,Desk Lamp,furniture,-1,52.16,0.15,Banja Luka,Cash on Delivery
8139,ORD15792,2025-05-29,CUST0167,Customer 0167,Notebook,office supplies,-1,6.05,0.00,Sarajevo,PayPal
9765,ORD15177,2025-06-27,CUST0213,Customer 0213,Wireless Mouse,electronics,-1,24.35,0.00,Sarajevo,Credit Card
10971,ORD06544,2025-07-19,CUST0436,Customer 0436,Office Chair,furniture,-1,190.08,0.10,Banja Luka,Credit Card
11128,ORD10457,2025-07-22,CUST0348,Customer 0348,Coffee Maker,home & kitchen,-1,87.16,0.00,Sarajevo,Bank Transfer
12677,ORD07472,2025-08-19,CUST0397,Customer 0397,Desk Lamp,furniture,-1,50.27,0.00,Banja Luka,Credit Card
14953,ORD05138,2025-09-30,CUST0128,Customer 0128,Yoga Mat,sports,-1,28.77,0.00,Brcko,PayPal
16753,ORD14930,2025-11-03,CUST0231,Customer 0231,Mechanical Keyboard,electronics,-1,80.36,0.15,Mostar,PayPal
19339,ORD01095,2025-12-19,CUST0128,Customer 0128,Air Fryer,home & kitchen,-1,123.70,0.05,Prijedor,Credit Card


Removing rows with negativ `quantity` values.

In [13]:
negative_quantity_rows = df[df['quantity'] < 0]
print("Rows with negative quantity:")
print(negative_quantity_rows)
# Removing negative quantity rows
df = df[df['quantity'] >= 0]
print("Shape after removing negative quantity rows:", df.shape)

Rows with negative quantity:
       order_id order_date customer_id  customer_name              product  \
1240   ORD02912 2025-01-22    CUST0022  Customer 0022         Skincare Set   
6302   ORD17212 2025-04-25    CUST0139  Customer 0139            Desk Lamp   
8139   ORD15792 2025-05-29    CUST0167  Customer 0167             Notebook   
9765   ORD15177 2025-06-27    CUST0213  Customer 0213       Wireless Mouse   
10971  ORD06544 2025-07-19    CUST0436  Customer 0436         Office Chair   
11128  ORD10457 2025-07-22    CUST0348  Customer 0348         Coffee Maker   
12677  ORD07472 2025-08-19    CUST0397  Customer 0397            Desk Lamp   
14953  ORD05138 2025-09-30    CUST0128  Customer 0128             Yoga Mat   
16753  ORD14930 2025-11-03    CUST0231  Customer 0231  Mechanical Keyboard   
19339  ORD01095 2025-12-19    CUST0128  Customer 0128            Air Fryer   

              category  quantity  unit_price  discount        city  \
1240            beauty        -1       40.

Removing negativ `unit_price` values.

In [14]:
negativ_unit_price_rows = df[df['unit_price'] < 0]
print("Rows with negative unit price:")
print(negativ_unit_price_rows)
# Removing negative unit price rows
df = df[df['unit_price'] >= 0]
print("Shape after removing negative unit price rows:", df.shape)

Rows with negative unit price:
       order_id order_date customer_id  customer_name              product  \
282    ORD00742 2025-01-06    CUST0341  Customer 0341              Printer   
1297   ORD08043 2025-01-24    CUST0054  Customer 0054         Coffee Maker   
6929   ORD14051 2025-05-07    CUST0046  Customer 0046            Air Fryer   
6944   ORD11347 2025-05-07    CUST0373  Customer 0373  Mechanical Keyboard   
15914  ORD14328 2025-10-17    CUST0370  Customer 0370           Smartphone   

              category  quantity  unit_price  discount        city  \
282    office supplies         2       -50.0      0.20    Prijedor   
1297    home & kitchen         3       -50.0      0.10       Tuzla   
6929    home & kitchen         3       -50.0      0.05  Banja Luka   
6944       electronics         1       -50.0      0.00       Tuzla   
15914      electronics         1       -50.0      0.00    Sarajevo   

      payment_method  
282      Credit Card  
1297     Credit Card  
6929      

From befor we saw, that datasrt has missing values in two columns, `city` and `payment_method`. We will make short review in it, but we won't remove this missing values just yet.

In [15]:
df[df["city"].isna()].head(10)

,order_id,order_date,customer_id,customer_name,product,category,quantity,unit_price,discount,city,payment_method
261,ORD06154,2025-01-06,CUST0090,Customer 0090,Bluetooth Speaker,electronics,2,62.55,0.05,NaN,Bank Transfer
1175,ORD03006,2025-01-21,CUST0401,Customer 0401,Coffee Maker,home & kitchen,1,91.36,0.05,NaN,Credit Card
1203,ORD16888,2025-01-22,CUST0483,Customer 0483,Office Chair,furniture,1,191.52,0.00,NaN,Credit Card
1277,ORD03649,2025-01-23,CUST0493,Customer 0493,Hair Dryer,beauty,2,67.70,0.00,NaN,Credit Card
1354,ORD06988,2025-01-25,CUST0409,Customer 0409,Desk Lamp,furniture,3,48.81,0.05,NaN,Credit Card
1386,ORD12720,2025-01-25,CUST0490,Customer 0490,Yoga Mat,sports,1,29.52,0.00,NaN,Credit Card
1398,ORD06126,2025-01-25,CUST0134,Customer 0134,Desk Lamp,furniture,2,48.13,0.05,NaN,Bank Transfer
1559,ORD14678,2025-01-28,CUST0467,Customer 0467,Skincare Set,beauty,3,40.49,0.00,NaN,Credit Card
1721,ORD10308,2025-01-31,CUST0429,Customer 0429,Notebook,office supplies,4,5.99,0.00,NaN,Credit Card
1726,ORD01657,2025-01-31,CUST0376,Customer 0376,Hair Dryer,beauty,1,70.17,0.15,NaN,Bank Transfer


In [16]:
df[df['payment_method'].isna()].head(10)

,order_id,order_date,customer_id,customer_name,product,category,quantity,unit_price,discount,city,payment_method
24,ORD14149,2025-01-01,CUST0161,Customer 0161,Skincare Set,beauty,1,41.01,0.05,Banja Luka,NaN
31,ORD06605,2025-01-01,CUST0282,Customer 0282,Notebook,office supplies,1,5.85,0.05,Bijeljina,NaN
127,ORD04490,2025-01-03,CUST0414,Customer 0414,Yoga Mat,sports,3,29.31,0.05,Tuzla,NaN
241,ORD07770,2025-01-05,CUST0223,Customer 0223,Smartphone,electronics,4,508.10,0.00,Trebinje,NaN
608,ORD16183,2025-01-12,CUST0263,Customer 0263,Wireless Mouse,electronics,1,23.90,0.00,Mostar,NaN
789,ORD19136,2025-01-15,CUST0259,Customer 0259,Coffee Maker,home & kitchen,4,87.27,0.05,Zenica,NaN
1197,ORD02512,2025-01-21,CUST0041,Customer 0041,Office Chair,furniture,3,186.76,0.05,Sarajevo,NaN
1586,ORD13751,2025-01-29,CUST0345,Customer 0345,Wireless Mouse,electronics,1,25.03,0.00,Zenica,NaN
2200,ORD04403,2025-02-10,CUST0221,Customer 0221,Office Chair,furniture,1,196.42,0.10,Mostar,NaN
2546,ORD10283,2025-02-16,CUST0248,Customer 0248,Wireless Mouse,electronics,2,24.54,0.05,Zenica,NaN


In [17]:
df[["city", "payment_method"]].isna().sum()

,0
city,100
payment_method,75
